# generate a dataset of examples with a decomposing prefix applied

In [ ]:
from pathlib import Path
import sys
import os

current_directory = Path(os.getcwd())
print(current_directory)
root_directory = current_directory.parent.parent
print(root_directory)

sys.path.append(str(root_directory))

In [ ]:
from pathlib import Path
import typing as t
import re
from pprint import pprint
import coq_serapy as c
import json
import pickle

from src.config import CONFIG
from src.coq_serapy_util import LemmaLocation, Coq
from src.dataset import Example, COQ_WIGDERSON_DEV_SAMPLED_DATASET, COQ_WIGDERSON_DEV_PERFECT_PREFIX

In [ ]:
for example in COQ_WIGDERSON_DEV_SAMPLED_DATASET:
    print(example.name)
    print(COQ_WIGDERSON_DEV_PERFECT_PREFIX[example.name])
    print()

In [ ]:
TEST_EXAMPLE = next(example for example in COQ_WIGDERSON_DEV_SAMPLED_DATASET if example.name == "coq-wigderson-coloring.v-phase2_color_bound")
TEST_EXAMPLE_PREFIX = COQ_WIGDERSON_DEV_PERFECT_PREFIX[TEST_EXAMPLE.name]

print(TEST_EXAMPLE.name)
pprint(TEST_EXAMPLE)
print(TEST_EXAMPLE_PREFIX)

In [ ]:
c = Coq(lemma_location=TEST_EXAMPLE.location)

In [ ]:
result = c.run_code(TEST_EXAMPLE.proposition_command + "\nProof.\n" + TEST_EXAMPLE_PREFIX)
result

In [ ]:
pprint(result.context.fg_goals)

In [ ]:
def prefix_for_idx(idx: int):
    # repeat 'admit.' idx times
    return ' admit. ' * idx

[TEST_EXAMPLE_PREFIX + prefix_for_idx(i) for i in range(len(result.context.fg_goals))]

In [ ]:
def make_subgoal_examples(example: Example, perfect_prefix: str) -> t.List[Example]:
    if perfect_prefix.strip() == '':
        return [example]
    
    c = Coq(lemma_location=example.location)
    result = c.run_code(example.proposition_command + "\nProof.\n" + perfect_prefix)
    if result.context is None:
        raise ValueError("No context found in result.")
    
    return [Example(
        location=example.location,
        proposition_command=example.proposition_command,
        gold_standard_proof = example.gold_standard_proof,
        hint = example.hint,
        proof_prefix=perfect_prefix + prefix_for_idx(i) + ' - ',
        tag=f"subgoal{i}"
    ) for i in range(len(result.context.fg_goals))]

In [ ]:
SUBGOAL_DATASET = [example for example in COQ_WIGDERSON_DEV_SAMPLED_DATASET for example in make_subgoal_examples(example, COQ_WIGDERSON_DEV_PERFECT_PREFIX[example.name])]

print(len(SUBGOAL_DATASET))

In [ ]:
for example in SUBGOAL_DATASET:
    print(example.name)

In [ ]:
# pickle the dev set to ./COQ_WIGDERSON_DEV_PERFECT_SUBGOALS.pkl
W_DEV_SAMPLED_FILE = current_directory / "../../data/COQ_WIGDERSON_DEV_PERFECT_SUBGOALS.pkl"
with open(W_DEV_SAMPLED_FILE, "wb") as f:
    pickle.dump(SUBGOAL_DATASET, f)